# IDDPM(Improved Denoising Diffusion Probabilistic Models)

IDDPM，顾名思义，就是对 DDPM 的工作进行了一系列的**改进**，在 DDPM 篇中，我已经详细介绍了 DDPM 的原理、代码、训练与推理，因此就不会有 DDPM 篇讲解的那么详细了

上一篇 DDPM 中我们可以知道，他的生成质量高，但是在负对数似然(NLL)上表现不够好，IDDPM 就让 DDPM 保证生成质量的同时，提升了 NLL 且加速了采样速度

本篇notebook基于如下链接的内容和代码进行整理，参考链接：

[参考知乎解析](https://zhuanlan.zhihu.com/p/706419495)

[IDDPM 论文](https://arxiv.org/pdf/2102.09672)

[IDDPM Pytorch 实现](https://github.com/LittleNyima/code-snippets/tree/master/iddpm-tutorial)


## 理论介绍

IDDPM 主要针对以下方面做出了改进:

- 不使用固定的方差，而是和 $\epsilon_\theta$ 一样，是可学习的；

- 噪声调度使用余弦形式，而非线性


### 可学习的方差

作者发现，在扩散的过程中，最初的几步扩散对 VLB(变分下界) 的影响是最大的，这些步里的 $\sigma_t$，也就是方差的选择就对分布拟合与似然优化依然具有显著影响，简单固定的方差就没办法充分优化这几步的损失，要理解这个，我们得知道 VLB 是什么

#### VLB(Variational Lower Bound，变分下界)
在 DDPM 中，因为即使不介绍这个东西也不影响核心思路的理解，因此我没有介绍，但是在这里，我们就得详细学习一下 VLB 了；

对数似然通俗一点理解，就是说，对于模型输出的图像x，似然$p(x)$越大，模型对真实数据的拟合就越好，越小则越差，因此最大化似然就是我们的目标；

我们想最大化$\log p(x)$，但是这个东西无法直接计算，但是我们有:

$$
\log p_\theta(x_0) \geq \mathbb{E}_{q\left(x_{1:T}  \mid x_0 \right)} \left[\log \frac{p_\theta(x_{0:T})}{q(x_{1:T \mid x_0})}\right]
$$

右边就是变分下界

对扩散过程做一系列复杂的数学操作，VLB 损失就变成了:

$$
\begin{aligned}
\mathcal{L}_{\text{VLB}} =
& D_{\mathrm{KL}}\big(q(x_T|x_0) \parallel p_\theta(x_T)\big) \\
&+ \sum_{t=2}^T D_{\mathrm{KL}}\big(q(x_{t-1}|x_t,x_0) \parallel p_\theta(x_{t-1}|x_t)\big) \\
&- \log p_\theta(x_0|x_1)
\end{aligned}
$$

我们逐项来看：

- $ D_{\mathrm{KL}}\big(q(x_T|x_0) \parallel p_\theta(x_T)\big)$：
意思是说，我们加噪最后得到的$x_T$要求尽可能接近$p_\theta(x_T)$，因为我们假设积累后的加噪结果仍然符合标准的高斯分布

- $ \sum_{t=2}^T D_{\mathrm{KL}}\big(q(x_{t-1}|x_t,x_0) \parallel p_\theta(x_{t-1}|x_t)\big)$：
意思是说，对于每一步模型从$x_t$预测得到的$x_{t-1}$(由$p_\theta(x_{t-1}|x_t)$得到)，要尽可能地和真实的$x_{t-1}$(由$q(x_{t-1}|x_t,x_0)$得到)相似，求和就是说要求每一步都尽可能满足这个要求，就是**下文图中对应纵坐标的含义**，每一个t对应了一个 KL 散度

- $- \log p_\theta(x_0|x_1)$：
意思是说，模型从只加噪了一次的图像$x_1$预测得到的原始图像$x_0$的似然


#### 可学习方差
理解了上面，我们就可以看下面的图:

<div align="center">
    <img src="./imgs/paper_fig2.png" alt="paper_fig2" style="width: 400px; height: auto;">
</div>

可以看到，前面几步的 KL 散度带来的损失是最大的，因此优化前面几步的 KL 散度损失，对提升整体的 VLB 帮助非常大

前面几步的噪声是很大的，选择一个好的方差，对于优化 KL 散度有非常大的帮助，因为这几步的 KL 散度误差对于方差敏感(详见下面)，因此作者认为可以选择比较好的方差来取得更好的对数似然 (log-likelihood)，就引入了一个可学习的方差:

$$
\Sigma_\theta(x_t, t) = \exp\left(v_t \log \beta_t + (1 - v_t) \log \tilde{\beta}_t\right)
$$

其中:

- $\beta_t$: 向前加噪过程的噪声调度

- $\tilde{\beta}_t$: 已知$x_0$时反向去噪的理论最优方差，由向前过程推导得到

- $v_t$: 模型输出的0~1之间的一个数，用于平衡上述二者

为什么前几步的 KL 散度对方差敏感呢？(这几步的 KL 散度给损失项的贡献也是最大的)

高斯 KL 公式如下:

$$
D_{KL} = \left( \mathcal{N}(\mu_q, \sigma_q^2) \parallel \mathcal{N}(\mu_\theta, \sigma_\theta^2) \right) = \frac{1}{2} \left( \log \frac{\sigma_\theta^2}{\sigma_q^2} + \frac{\sigma_q^2 + (\mu_q - \mu_\theta)^2}{\sigma_\theta^2} - 1 \right)
$$

在前几步中，$(\mu_q - \mu_\theta)^2$比较大，因此方差$\sigma_\theta^2$的微小变化也会导致整个式子变化很大，因此选择合适的$\sigma^2$就尤为关键了，所以作者就把选择的任务交给模型来做，让模型选择合适的$\sigma^2$，从而降低前几项主导的 KL 散度误差项，从而让整体的 **VLB** 变大


In [1]:
import torch 
import numpy as np


In [2]:
# KL散度公式实现
def normal_kl(mean1, logvar1, mean2, logvar2):
    """
    Compute the KL divergence between two gaussians.

    Shapes are automatically broadcasted, so batches can be compared to
    scalars, among other use cases.
    """
    tensor = None
    for obj in (mean1, logvar1, mean2, logvar2):
        if isinstance(obj, torch.Tensor):
            tensor = obj
            break
    assert tensor is not None, "at least one argument must be a Tensor"

    # Force variances to be Tensors. Broadcasting helps convert scalars to
    # Tensors, but it does not work for th.exp().
    logvar1, logvar2 = [
        x if isinstance(x, torch.Tensor) else torch.tensor(x).to(tensor)
        for x in (logvar1, logvar2)
    ]

    return 0.5 * (
        -1.0
        + logvar2 - logvar1  # log(σ2²/σ1²)
        + torch.exp(logvar1 - logvar2)  # σ1² / σ2²
        + ((mean1 - mean2) ** 2) * torch.exp(-logvar2)   # (μ1-μ2)² / σ2²
    )


In [3]:
mean1 = torch.tensor([0.0])
logvar1 = torch.tensor([0.0])
mean2 = torch.tensor([0.0])
logvar2 = torch.tensor(0.0)
print(f'相同高斯分布的KL散度: {normal_kl(mean1, logvar1, mean2, logvar2)}')

mean2 = torch.tensor([1.0])
print(f'N(0,1)与N(1,1): {normal_kl(mean1, logvar1, mean2, logvar2)}')


相同高斯分布的KL散度: tensor([0.])
N(0,1)与N(1,1): tensor([0.5000])


可学习方差在采样中的构造核心代码:
```python
# [bs, 6, H, W]
preds = self.pred_model(batch_xt, t_inference)  
# 后三个通道是logvar相关的系数
pred_vars = preds[:, :3, :, :]  
# 公式里的 \bar\beta 的计算
min_log = gather(posterior_log_variance_clipped, t_inference)  
# \beta
max_log = gather(torch.log(betas), t_inference)  
# v(就是frac) = (model_out+1)/2
frac = (pred_vars + 1.0) / 2.0  
log_variance = frac * max_log + (1.0 - frac) * min_log
# 最终标准差
stddev = torch.exp(0.5 * log_variance)
```

### 余弦方差调度
作者引入了余弦方差调度替换原先的线性调度:

$$
\bar\alpha_t = \frac{f(t)}{f(0)}, 
f(t) = \cos^2 \left( \frac{t/T + s}{1+s} \cdot \frac{\pi}{2} \right)
$$

可以看一下原论文中的对比图:

<div align="center">
    <img src="./imgs/paper_fig5.png" alt="paper_fig5" style="width: 400px; height: auto;">
</div>

纵坐标可以理解成信号的保留程度，可以看到，原始 DDPM 线性调度在早期会让噪声迅速淹没原始图像，接着很快就接近0了，不利于模型的学习，而 IDDPM 的余弦方差调度会让原图像信号线性衰减，且衰减更慢，从而能够让模型更好进行学习

我们可以对比一下二者(同1000步加噪)


In [4]:
from c_iddpm import vis_img_change_cos
import math


def get_cos_noise_scheduler(n_steps=1000, max_beta=0.999, s=0.008) -> np.ndarray:
    """
    余弦噪声调度
    
    :param n_steps: 总扩散步数
    :param max_beta: beta最大值
    :param s: 计算f(t)用到的平滑值s
    :return: 余弦噪声调度
    """
    # f(t) = cos((t/T+s)/(1+s) * (pi/2))^2，这里t/T就是t_norm
    # f(t)其实就是alpha_bar[t]
    ft = lambda t_norm: math.cos(((t_norm+s)/(1+s)) * (math.pi/2)) ** 2

    betas = []
    for i in range(n_steps):
        t1 = i/n_steps
        t2 = (i+1)/n_steps
        # beta_t = 1 - f(t+1)/t(t)，不能超过max_beta
        betas.append(min(1 - ft(t2)/ft(t1), max_beta))

    return torch.tensor(betas, dtype=torch.float32)


vis_img_change_cos(origin_img_path='./imgs/gugugaga.png',
               vis_interval=50, 
               get_cos_nosise_scheduler_func=get_cos_noise_scheduler,
               output_dir='./imgs',
               gif_name='gugugaga_noising_cos.gif') 


GIF saved to: imgs\gugugaga_noising_cos.gif


我们可以对比一下，可以看到，余弦调度的会比线性调度加噪更慢

<div style="display: flex; justify-content: center; gap: 20px; align-items: center;">

  <div style="text-align: center;">
    <img src="./imgs/gugugaga_noising.gif" alt="linear_noising" style="width: 250px; height: auto;">
    <p style="margin: 4px 0 0; font-weight: bold;">线性调度（DDPM）</p>
  </div>

  <div style="text-align: center;">
    <img src="./imgs/gugugaga_noising_cos.gif" alt="cosine_noising" style="width: 250px; height: auto;">
    <p style="margin: 4px 0 0; font-weight: bold;">余弦调度（IDDPM）</p>
  </div>

</div>

### 损失函数
IDDPM 最终使用的损失函数是这样的:

$$
\mathcal{L}_{total} = \mathcal{L}_{simple} + \lambda \mathcal{L}_{vlb}
$$

而$\mathcal{L}_{vlb}$就如我们上面看到的那样:

$$
\begin{aligned}
\mathcal{L}_{\text{VLB}} =
& D_{\mathrm{KL}}\big(q(x_T|x_0) \parallel p_\theta(x_T)\big) \\
&+ \sum_{t=2}^T D_{\mathrm{KL}}\big(q(x_{t-1}|x_t,x_0) \parallel p_\theta(x_{t-1}|x_t)\big) \\
&- \log p_\theta(x_0|x_1)
\end{aligned}
$$

为了防止优化 VLB 时影响了原始 DDPM 的损失函数，作者采用的权重较小，且阻止了 VLB 中$\mu_\theta(x_t, t)$的梯度传递，让$\mathcal{L}_{simple}$主导均值的梯度


In [5]:
def training_losses(self, x0: torch.Tensor, xt: torch.Tensor, t: torch.Tensor,
                        noises: torch.Tensor, vlb_weight: float = 1e-3):
    _, channels, _, _ = x0.shape 
    pred = self.pred_model(xt, t)
    pred_noises, pred_vars = torch.split(pred, channels, dim=1)

    # L_simple，和DDPM一致
    l_simple = (pred_noises - noises) ** 2
    l_simple = l_simple.mean(dim=list(range(1, len(l_simple.shape))))

    # L_vlb，pred_noises.detach()阻断vlb项中预测均值的梯度回传
    l_vlb = self.vlb_loss(pred_noises.detach(), pred_vars, x0, xt, t)

    return l_simple + vlb_weight * l_vlb


### 重要性采样
由于不同时间步产生的 VLB 损失大小不一，均匀采样并不能很好地学习每一步的去噪过程，应该给予损失较大的时间步更加频繁地采样学习，从而更好地进行模型的训练，因此作者引入了**重要性采样**:

$$
\mathcal{L}_{vlb} = E_{t \sim p_t} \left[ \frac{\mathcal{L}_t}{p_t} \right], 
\quad \text{where } p \propto \sqrt{E\left[\mathcal{L}_t^2\right]} \text{ and } \sum p_t = 1
$$

上面的式子很复杂，我们直接看作者怎么实现的，就是说，训练的时候每个 t 产生的损失都保存最近的10个且会动态进行更新，在计算每个 t 的采样概率时，先求出每个t的总 loss 进行归一化，那么就得到了每个 t 的采样概率；

$$
\text{prob\_sample\_t} = 
\frac{\sum_{i=1}^{10} \text{history\_t\_loss}_i}{\sum_{j=1}^T \sum_{i=1}^{10} \text{history\_j\_loss}_i}
$$

因为采样会偏向于多采样比较大的损失项对应的 t，为了保证损失无偏，就需要给 total_loss 计算时，每一个 t 的损失项加权，T为总扩散步数

$$
\text{weight\_t} = \frac{1}{T \cdot \text{prob\_sample\_t}}
$$

计算总损失时就变成了(T_sample 为一次训练的 batch 数):

$$
\text{total\_loss} = \sum_{i=1}^{\text{T\_sample}} \text{weight\_t} \cdot 
\text{loss\_term\_t}
$$


In [6]:
class ImportantSampler:
    """
    重要性采样实现
    """
    def __init__(self, iddpm_model, history_per_term=10, uniform_prob=0.001):
        """
        :param iddpm_model: 扩散模型
        :param history_per_term: 每个t保存最近的loss数
        :param uniform_prob: 采用均匀采样的概率
        """
        self.iddpm = iddpm_model
        self.history_per_term = history_per_term
        self.uniform_prob = uniform_prob

        # 记录每个t产生的loss，以及每个t已记录的loss个数
        self._loss_history = np.zeros([self.iddpm.n_steps, self.history_per_term], 
                                      dtype=np.float64)
        self._loss_cnt = np.zeros([self.iddpm.n_steps], dtype=int)

    def weights(self):
        """更新每个t的权重"""
        if not self._warm_up():  # 若没有收集足够多的loss，均匀采样
            return np.ones([self.iddpm.n_steps], dtype=np.float64)
        
        # 求出每个t的loss^2的均值 -> [n_steps, 1]
        weights = np.sqrt(np.mean(self._loss_history ** 2, axis=-1))
        # 归一化 weights
        weights /= np.sum(weights)
        # final_weights = (1-ε)*loss_weights + ε*uniform_weights
        weights *= 1 - self.uniform_prob  # (1-ε)*loss_weights
        weights += self.uniform_prob / len(weights)  # + ε*uniform_weights

        return weights

    def _warm_up(self):
        return (self._loss_cnt == self.history_per_term).all()
    
    def update_with_all_losses(self, ts, losses):
        """t的历史损失更新"""
        for t, loss in zip(ts, losses):
            if self._loss_cnt[t] == self.history_per_term:
                # 移除最早加入的loss
                self._loss_history[t, :-1] = self._loss_history[t, 1:]
                self._loss_history[t, -1] = loss
            else:
                self._loss_history[t, self._loss_cnt[t]] = loss
                self._loss_cnt[t] += 1

    def sample(self, batch_size, device):
        """重要性采样"""
        # 概率 -> # [n_steps, 1]
        w = self.weights()  
        p = w / np.sum(w)

        # 按概率选择bs个t
        indices_np = np.random.choice(len(p), size=(batch_size,), p=p)
        indices = torch.from_numpy(indices_np).long().to(device)
        
        # loss_weight = 1/(n_steps*p[t])，让频繁抽取到的t对应的loss权重降低
        # 用于后续给损失加权，防止模型偏向训练频繁采样的t
        loss_weights_np = 1 / (len(p) * p[indices_np])
        loss_weights = torch.from_numpy(loss_weights_np).float().to(device)
        
        return indices, loss_weights
    

### 似然的估计(可以跳过)

#### 近似计算标准正态分布累积分布
公式如下:

$$
\Phi(x) \approx 
\frac{1}{2}\left(
1 + \tanh\left(
\sqrt{\frac{2}{\pi}} \left( x + 0.044715\,x^3 \right)
\right)
\right)
$$

#### 离散化的高斯分布，对数似然的计算
公式如下:

$$
\begin{aligned}
x_{\pm} &= \frac{(x - \mu) \pm 1/255}{\sigma}, \\
\log p(x) &=
\begin{cases}
\log\Phi(x_{+}), & x < -0.999, \\
\log(1-\Phi(x_{-})), & x > 0.999, \\
\log(\Phi(x_{+})-\Phi(x_{-})), & \text{otherwise}.
\end{cases}
\end{aligned}
$$

你可以这么理解，计算的似然就是真实的$x_0$的每个像素，在模型输出的高斯分布下的对数概率，概率越高，模型觉得这个像素的值越合理，训练时就是要让最后一步预测的高斯分布下，真实数据$x_0$的出现更加合理，也就更贴近真实数据的分布，从而让模型更好的学习真实数据的分布，**输出的负数是用于损失项的**

In [9]:
def approx_standard_normal_cdf(x):
    """
    A fast approximation of the cumulative distribution function of the
    standard normal.
    """
    return 0.5 * (1.0 + torch.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * torch.pow(x, 3))))

def discretized_gaussian_log_likelihood(x, *, means, log_scales):
    """
    Compute the log-likelihood of a Gaussian distribution discretizing to a
    given image.

    :param x: the target images. It is assumed that this was uint8 values,
              rescaled to the range [-1, 1].
    :param means: the Gaussian mean Tensor.
    :param log_scales: the Gaussian log stddev Tensor.
    :return: a tensor like x of log probabilities (in nats).
    """
    assert x.shape == means.shape == log_scales.shape
    centered_x = x - means
    inv_stdv = torch.exp(-log_scales)
    plus_in = inv_stdv * (centered_x + 1.0 / 255.0)
    cdf_plus = approx_standard_normal_cdf(plus_in)
    min_in = inv_stdv * (centered_x - 1.0 / 255.0)
    cdf_min = approx_standard_normal_cdf(min_in)
    log_cdf_plus = torch.log(cdf_plus.clamp(min=1e-12))
    log_one_minus_cdf_min = torch.log((1.0 - cdf_min).clamp(min=1e-12))
    cdf_delta = cdf_plus - cdf_min
    log_probs = torch.where(
        x < -0.999,
        log_cdf_plus,
        torch.where(x > 0.999, log_one_minus_cdf_min, torch.log(cdf_delta.clamp(min=1e-12))),
    )
    assert log_probs.shape == x.shape
    
    return log_probs


In [ ]:
x = torch.tensor([-0.9995, -0.5, 0.0, 0.5, 0.9995])
means = torch.zeros_like(x)
log_scales = torch.zeros_like(x) 

log_probs = discretized_gaussian_log_likelihood(
    x, means=means, log_scales=log_scales
)

print(f'x: {x}')
print(f'means, log_scales: {means} {log_scales}')
print(f'对数似然: {log_probs}')


x: tensor([-0.9995, -0.5000,  0.0000,  0.5000,  0.9995])
means, log_scales: tensor([0., 0., 0., 0., 0.]) tensor([0., 0., 0., 0., 0.])
对数似然: tensor([-1.8333, -5.8926, -5.7670, -5.8926, -1.8333])


## 完整的 IDDPM 实现

In [ ]:
"""
Thanks to:
https://github.com/openai/improved-diffusion/blob/main/improved_diffusion/gaussian_diffusion.py
https://github.com/LittleNyima/code-snippets/blob/master/iddpm-tutorial/iddpm_training.py
"""
import numpy as np
import torch
import torch.nn as nn
import math
from typing import Tuple, Optional
from tqdm import tqdm
from c_iddpm.loss_utils import normal_kl, discretized_gaussian_log_likelihood
from PIL import Image
import os


def gather(consts: torch.Tensor, t: torch.Tensor):
    """Gather consts for $t$ and reshape to feature map shape"""
    c = consts.gather(-1, t)  # 取出t对应的噪声调度、α或α的连乘
    
    # -> [bs, 1, 1, 1]，运算时广播为[bs, C, H, W]
    return c.reshape(-1, 1, 1, 1)

def get_cos_noise_scheduler(n_steps=1000, max_beta=0.999, s=0.008) -> np.ndarray:
    """
    余弦噪声调度
    
    :param n_steps: 总扩散步数
    :param max_beta: beta最大值
    :param s: 计算f(t)用到的平滑值s
    :return: 余弦噪声调度
    """
    # f(t) = cos((t/T+s)/(1+s) * (pi/2))^2，这里t/T就是t_norm
    # f(t)其实就是alpha_bar[t]
    ft = lambda t_norm: math.cos(((t_norm+s)/(1+s)) * (math.pi/2)) ** 2

    betas = []
    for i in range(n_steps):
        t1 = i/n_steps
        t2 = (i+1)/n_steps
        # beta_t = 1 - f(t+1)/t(t)，不能超过max_beta
        betas.append(min(1 - ft(t2)/ft(t1), max_beta))

    return torch.tensor(betas, dtype=torch.float32)

class ImportantSampler:
    """
    重要性采样实现
    """
    def __init__(self, iddpm_model, history_per_term=10, uniform_prob=0.001):
        """
        :param iddpm_model: 扩散模型
        :param history_per_term: 每个t保存最近的loss数
        :param uniform_prob: 采用均匀采样的概率
        """
        self.iddpm = iddpm_model
        self.history_per_term = history_per_term
        self.uniform_prob = uniform_prob

        # 记录每个t产生的loss，以及每个t已记录的loss个数
        self._loss_history = np.zeros([self.iddpm.n_steps, self.history_per_term], 
                                      dtype=np.float64)
        self._loss_cnt = np.zeros([self.iddpm.n_steps], dtype=int)

    def weights(self):
        """更新每个t的权重"""
        if not self._warm_up():  # 若没有收集足够多的loss，均匀采样
            return np.ones([self.iddpm.n_steps], dtype=np.float64)
        
        # 求出每个t的loss^2的均值 -> [n_steps, 1]
        weights = np.sqrt(np.mean(self._loss_history ** 2, axis=-1))
        # 归一化 weights
        weights /= np.sum(weights)
        # final_weights = (1-ε)*loss_weights + ε*uniform_weights
        weights *= 1 - self.uniform_prob  # (1-ε)*loss_weights
        weights += self.uniform_prob / len(weights)  # + ε*uniform_weights

        return weights

    def _warm_up(self):
        return (self._loss_cnt == self.history_per_term).all()
    
    def update_with_all_losses(self, ts, losses):
        """t的历史损失更新"""
        for t, loss in zip(ts, losses):
            if self._loss_cnt[t] == self.history_per_term:
                # 移除最早加入的loss
                self._loss_history[t, :-1] = self._loss_history[t, 1:]
                self._loss_history[t, -1] = loss
            else:
                self._loss_history[t, self._loss_cnt[t]] = loss
                self._loss_cnt[t] += 1

    def sample(self, batch_size, device):
        """重要性采样"""
        # 概率 -> # [n_steps, 1]
        w = self.weights()  
        p = w / np.sum(w)

        # 按概率选择bs个t
        indices_np = np.random.choice(len(p), size=(batch_size,), p=p)
        indices = torch.from_numpy(indices_np).long().to(device)
        
        # loss_weight = 1/(n_steps*p[t])，让频繁抽取到的t对应的loss权重降低
        # 用于后续给损失加权，防止模型偏向训练频繁采样的t
        loss_weights_np = 1 / (len(p) * p[indices_np])
        loss_weights = torch.from_numpy(loss_weights_np).float().to(device)
        
        return indices, loss_weights

class IDDPM(nn.Module):
    """
    IDDPM类
    """
    def __init__(self, model: nn.Module, n_steps: int):
        super().__init__()

        self.device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
        self.pred_model = model.to(self.device)  # output -> [bs, 2*image_channels, H, W]
        self.n_steps = n_steps
        
        """
        betas: 余弦噪声调度
        alphas: 1 - betas
        alphas_cumprod: alpha连乘
        alpha_cumprod_prev: 得到alpha_bar_{t-1}
        alpha_cumprod_next: 得到alpha_bar_{t+1}
        timesteps: 去噪步 T-1 -> 0
        """
        self.betas = get_cos_noise_scheduler(n_steps=self.n_steps).to(self.device)
        self.alphas = (1 - self.betas).to(self.device)
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0).to(self.device)
        self.alphas_cumprod_prev = torch.concat((torch.ones(1).to(self.alphas_cumprod), self.alphas_cumprod[:-1])).to(self.device)
        self.alphas_cumprod_next = torch.concat((self.alphas_cumprod[1:], torch.zeros(1).to(self.alphas_cumprod))).to(self.device)
        self.timesteps = torch.arange(self.n_steps - 1, -1, -1).to(self.device)
    
    def q_xt_x0(self, x0: torch.Tensor, t: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        求出x_t的分布的均值和方差
        """
        # [gather](utils.html) $\alpha_t$ and compute $\sqrt{\bar\alpha_t} x_0$
        mean = gather(self.alphas_cumprod, t) ** 0.5 * x0
        # $(1-\bar\alpha_t) \mathbf{I}$
        var = 1 - gather(self.alphas_cumprod, t)
        
        return mean, var

    def q_sample(self, x0: torch.Tensor, t: torch.Tensor, eps: Optional[torch.Tensor] = None):
        """
        求出x_0加噪t步后的图像x_t
        """
        # $\epsilon \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$
        if eps is None:
            eps = torch.randn_like(x0)

        # get $q(x_t|x_0)$
        mean, var = self.q_xt_x0(x0, t)
        
        # Sample from $q(x_t|x_0)$
        return mean + (var ** 0.5) * eps
    
    def pred_mean_logvar(self, pred_noises: torch.Tensor, pred_vars: torch.Tensor,
                         batch_xt: torch.Tensor, t_inference: torch.Tensor):
        """
        根据(x_t, t)预测x_{t-1}的均值和标准差对数
        """
        # --------------------------------------------------------------------------------------------------
        # 参数预计算
        # --------------------------------------------------------------------------------------------------
        betas = self.betas.to(self.device)
        alphas = self.alphas.to(self.device)
        alphas_cumprod = self.alphas_cumprod.to(self.device)
        alphas_cumprod_prev = self.alphas_cumprod_prev.to(self.device)

        sqrt_recip_alphas_cumprod = (1.0 / alphas_cumprod) ** 0.5
        sqrt_recipm1_alphas_cumprod = (1.0 / alphas_cumprod - 1.0) ** 0.5
        posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)
        posterior_log_variance_clipped = torch.log(torch.concat((posterior_variance[1:2], posterior_variance[1:])))
        posterior_mean_coef1 = betas * alphas_cumprod_prev ** 0.5 / (1.0 - alphas_cumprod)
        posterior_mean_coef2 = (1.0 - alphas_cumprod_prev) * alphas ** 0.5 / (1.0 - alphas_cumprod)


        # 计算均值，与DDPM相同，只不过写法不同而已
        x_0 = gather(sqrt_recip_alphas_cumprod, t_inference)*batch_xt - \
                gather(sqrt_recipm1_alphas_cumprod, t_inference)*pred_noises
        mean = gather(posterior_mean_coef1, t_inference)*x_0.clamp(-1, 1) + \
                gather(posterior_mean_coef2, t_inference)*batch_xt
        
        # 计算方差
        min_log = gather(posterior_log_variance_clipped, t_inference)  # 公式里的 \bar\beta
        max_log = gather(torch.log(betas), t_inference)  # \beta
        frac = (pred_vars + 1.0) / 2.0  # v(就是frac) = (model_out+1)/2
        log_variance = frac * max_log + (1.0 - frac) * min_log

        return mean, log_variance

    @torch.no_grad()
    def p_sample(self, batch_size: int, in_channels: int, img_size: int, save_dir="./imgs"):
        """
        从标准高斯噪声采样，并采样去噪
        """
        batch_xt = torch.randn(batch_size, in_channels, img_size, img_size, device=self.device)
        timesteps = self.timesteps.to(self.device)

        for t in tqdm(timesteps, desc="Sampling"):
            t_inference = batch_xt.new_full((batch_size,), t, dtype=torch.long)  # -> [bs]
            preds = self.pred_model(batch_xt, t_inference)
            pred_noises, pred_vars = torch.split(preds, in_channels, dim=1)

            mean, log_variance = self.pred_mean_logvar(pred_noises, pred_vars, batch_xt, t_inference)

            # 学习的方差
            if t > 0:
                stddev = torch.exp(0.5 * log_variance)  # -> 最终标准差
            else:
                # 最后一步去噪方差设为0
                stddev = torch.zeros_like(mean)

            epsilon = torch.randn_like(batch_xt)
            batch_xt = mean + stddev*epsilon

        # range (-1, 1) -> (0, 1)
        # permute: [bs, C, H, W] -> [bs, H, W, C]，从而正常保存图像
        batch_imgs = (batch_xt / 2.0 + 0.5).clamp(0, 1).cpu().permute(0, 2, 3, 1).numpy()

        for i in range(batch_size):
            img = Image.fromarray((batch_imgs[i] * 255).astype('uint8'))
            img.save(os.path.join(save_dir, f"sample_{i}.png"))

        print(f'Saved all images at: {save_dir}')

        # return batch_imgs
    
    def get_true_log_var(self, x0: torch.Tensor, xt: torch.Tensor, t: torch.Tensor):
        """
        根据x0, xt获得真实x_{t-1}的均值和标准差对数
        """
        # --------------------------------------------------------------------------------------------------
        # 参数预计算
        # --------------------------------------------------------------------------------------------------
        betas = self.betas.to(self.device)
        alphas = self.alphas.to(self.device)
        alphas_cumprod = self.alphas_cumprod.to(self.device)
        alphas_cumprod_prev = self.alphas_cumprod_prev.to(self.device)

        posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)
        posterior_log_variance_clipped = torch.log(torch.concat((posterior_variance[1:2], posterior_variance[1:])))
        posterior_mean_coef1 = betas * alphas_cumprod_prev ** 0.5 / (1.0 - alphas_cumprod)
        posterior_mean_coef2 = (1.0 - alphas_cumprod_prev) * alphas ** 0.5 / (1.0 - alphas_cumprod)


        posterior_mean = gather(posterior_mean_coef1, t) * x0 + gather(posterior_mean_coef2, t) * xt
        posterior_log_variance_clipped = gather(posterior_log_variance_clipped, t)

        return posterior_mean, posterior_log_variance_clipped
    
    def vlb_loss(self, pred_noises: torch.Tensor, pred_vars: torch.Tensor,
                 x0: torch.Tensor, xt: torch.Tensor, t: torch.Tensor):
        """
        IDDPM的L_vlb
        """
        pred_mean, pred_logvar = self.pred_mean_logvar(pred_noises, pred_vars, xt, t)
        true_mean, true_logvar = self.get_true_log_var(x0, xt, t)

        # KL散度损失 [bs, C, H, W] -> [bs]
        # 注: /log(2.0)，是为了把e为底变为x为底
        # log2(x) = log(x)/log(2)
        kl = normal_kl(true_mean, true_logvar, pred_mean, pred_logvar)
        kl = kl.mean(dim=list(range(1, len(kl.shape)))) / math.log(2.0)

        # 负对数似然损失 [bs, C, H, W] -> [bs]
        # nll = discretized_gaussian_log_likelihood(x0, pred_mean, pred_logvar * 0.5)
        nll = discretized_gaussian_log_likelihood(x0, means=pred_mean, log_scales=pred_logvar * 0.5)
        nll = nll.mean(dim=list(range(1, len(nll.shape)))) / math.log(2.0)

        # 最后一步去噪用NLL，其他则用KL
        results = torch.where(t == 0, nll, kl)

        return results
    
    def training_losses(self, x0: torch.Tensor, xt: torch.Tensor, t: torch.Tensor,
                        noises: torch.Tensor, vlb_weight: float = 1e-3):
        _, channels, _, _ = x0.shape 
        pred = self.pred_model(xt, t)
        pred_noises, pred_vars = torch.split(pred, channels, dim=1)

        # L_simple，和DDPM一致
        l_simple = (pred_noises - noises) ** 2
        l_simple = l_simple.mean(dim=list(range(1, len(l_simple.shape))))
    
        # L_vlb，pred_noises.detach()阻断vlb项中预测均值的梯度回传
        l_vlb = self.vlb_loss(pred_noises.detach(), pred_vars, x0, xt, t)

        return l_simple + vlb_weight * l_vlb


In [7]:
# 500步训练展示
from c_iddpm import get_config, get_iddpm, IDDPM_trainer


config = get_config('./configs/iddpm_unet_config.yaml')
IDDPM_model = get_iddpm(config, False)
trainer = IDDPM_trainer(IDDPM_model, config)

trainer.train()


Training: 100%|██████████| 500/500 [01:29<00:00,  5.58it/s]


In [8]:
# 50000训练步的ckpt效果展示
from c_iddpm import get_config, get_iddpm


config = get_config('./configs/iddpm_unet_config.yaml')
IDDPM_model = get_iddpm(config, True)
IDDPM_model.p_sample(4, 3, 64)


Loaded checkpoint from ckpts\iddmp_unet_50000iter.pth


Sampling: 100%|██████████| 1000/1000 [00:50<00:00, 19.75it/s]

Saved all images at: ./imgs


<div style="display: flex; justify-content: center; gap: 12px; align-items: center; margin: 16px 0;">
  <div style="flex: 1; max-width: 120px; text-align: center;">
    <img src="./imgs/sample_0.png" style="width: 100%; height: auto;" alt="img1">
    <p style="margin: 4px 0 0; font-size: 14px;">图 1</p>
  </div>
  <div style="flex: 1; max-width: 120px; text-align: center;">
    <img src="./imgs/sample_1.png" style="width: 100%; height: auto;" alt="img2">
    <p style="margin: 4px 0 0; font-size: 14px;">图 2</p>
  </div>
  <div style="flex: 1; max-width: 120px; text-align: center;">
    <img src="./imgs/sample_2.png" style="width: 100%; height: auto;" alt="img3">
    <p style="margin: 4px 0 0; font-size: 14px;">图 3</p>
  </div>
  <div style="flex: 1; max-width: 120px; text-align: center;">
    <img src="./imgs/sample_3.png" style="width: 100%; height: auto;" alt="img4">
    <p style="margin: 4px 0 0; font-size: 14px;">图 4</p>
  </div>
</div>